# Speech emotion recognition with CNN from scratch

We train a **convolutional network built from scratch** (no pre-trained weights) to classify
the mel-spectrograms produced by `prepare_datasets.ipynb`
(folder `data/spectrograms/<emotion>/*.png`, Berlin EmoDB corpus). It is the counterpart of
`transfer_learning.ipynb`

The data comes from `usable_data/`, produced by the train/test split of `app.py`
(`streamlit run app.py`, then "Lancer le split train/test"):

    usable_data/train/labels.csv + <emotion>/*.png
    usable_data/test/labels.csv  + <emotion>/*.png

So this notebook works with any dataset or mix of datasets
The model is four blocks of two 3×3 convolutions + BatchNorm + ReLU + 2×2
max-pool, channels 1 → 32 → 64 → 128 → 256, then global average pooling, dropout and one
linear layer (~1.2 M parameters). Spectrograms are used in grayscale a
spectrogram is intrinsically a single time × frequency matrix, so no ImageNet normalisation
and no RGB, unlike the transfer-learning notebook.

In [ ]:
# deps (kernel "Python (CNN_RO11 .venv)")
# Same logic as transfer_learning.ipynb: install the CPU build of torch on a machine
# without a GPU, the CUDA build otherwise. If torch is already there in the right build,
# reinstall nothing.
import importlib.metadata as md
import shutil

gpu = shutil.which("nvidia-smi") is not None
try:
    installed = md.version("torch")
except md.PackageNotFoundError:
    installed = None
print(f"NVIDIA GPU: {gpu}   torch installed: {installed}")

wanted_cuda = gpu
have_cuda = bool(installed) and "+cu" in installed

if installed is None or wanted_cuda != have_cuda:
    print("-> (re)installing the", "CUDA" if wanted_cuda else "CPU-only", "build of torch")
    if wanted_cuda:
        %pip install -q -U --force-reinstall torch torchvision
    else:
        %pip install -q -U --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cpu
    print("!! RESTART THE KERNEL before running the next cell !!")
else:
    print("-> the installed build matches the machine, nothing to do")

%pip install -q scikit-learn matplotlib pillow pandas

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Make random operations deterministic (for reproducibility)
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

USABLE = Path("usable_data")       # output of the app.py split
IMG_SIZE = 128                     # spectrograms resized to 128 x 128
BATCH = 32
EPOCHS = 30
LR = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| CUDA build:", torch.version.cuda)
print("device:", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")
if device.type == "cpu":
    print("!! running on CPU: select the kernel \".venv (3.14.4)\" (the project .venv, CUDA build of torch)")
PIN = device.type == "cuda"   # page-locked host memory -> faster CPU -> GPU copies
torch.backends.cudnn.benchmark = PIN   # fixed input size: let cuDNN pick the fastest conv algorithms

### 1. Image inventory (train / test split made by `app.py`)

The dataset is now described by a metadata CSV, `data/spectrograms/labels.csv`, written
by the `prepare_*.ipynb` notebooks (one row per 3-second spectrogram). We read the label and
the speaker from the CSV, not from the file names: with several corpora the file-name
conventions differ, so speaker is a global integer id assigned in the CSV.

The split is by speaker: we hold out a fraction of the speaker ids for validation, so a
given speaker (and all its 3-second segments) is either in training or in validation, never
both.

In [ ]:
# The labels.csv files are the reference, not the folder content: usable_data/ is not
# emptied when the split is re-run, so it can contain images from a previous split.
def load_split(name):
    split_dir = USABLE / name
    df = pd.read_csv(split_dir / "labels.csv", na_values=["None"])
    df["path"] = [split_dir / s for s in df["spectrogram"]]
    df["label"] = [Path(s).parent.name for s in df["spectrogram"]]  # "<emotion>/<file>.png"
    return df


train_df = load_split("train")
val_df = load_split("test")

CLASSES = sorted(train_df["label"].unique())          # labels (emotion names) 
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}  # class name -> index

unknown = ~val_df["label"].isin(CLASSES)
if unknown.any():
    print(f"!! {unknown.sum()} test images with an emotion absent from train, dropped:",
          dict(Counter(val_df.loc[unknown, "label"])))
    val_df = val_df[~unknown]

shared = set(train_df["speaker"]) & set(val_df["speaker"])
if shared:
    print(f"!! {len(shared)} speakers are in both train and test: the validation score will be "
          "over-optimistic (split by speaker in app.py)")

# items : list of tuples (image_path, class_index, speaker_id) for each image
train_items = [(p, CLASS_TO_IDX[l], s) for p, l, s in zip(train_df["path"], train_df["label"], train_df["speaker"])]
val_items = [(p, CLASS_TO_IDX[l], s) for p, l, s in zip(val_df["path"], val_df["label"], val_df["speaker"])]

DATASETS = sorted(set(train_df["dataset"]) | set(val_df["dataset"]))
MODEL_PATH = f"cnn_scratch_{'_'.join(DATASETS).lower()}.pt"

print("datasets:", DATASETS)
print("classes :", CLASSES)
print(f"train {len(train_items)} images, {train_df['speaker'].nunique()} speakers / "
      f"val {len(val_items)} images, {val_df['speaker'].nunique()} speakers")
print("train distribution:", {CLASSES[i]: n for i, n in sorted(Counter(y for _, y, _ in train_items).items())})
print("val distribution  :", {CLASSES[i]: n for i, n in sorted(Counter(y for _, y, _ in val_items).items())})

### 2. Dataset and transforms

Each image is converted to grayscale, resized, turned into a tensor and
normalised around 0 (mean 0.5, std 0.5). Unlike transfer learning, no ImageNet
normalisation here: our network learns from scratch, it does not expect a particular
input distribution.

Augmentation (training only): `RandomErasing`, we mask a band
of time or frequency to make the network robust. No horizontal flip: reversing time is
not a plausible spectrogram.

In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.12), ratio=(0.3, 3.3)),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])


class SpectrogramDataset(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label, _ = self.items[i]
        return self.tf(Image.open(path)), label


train_loader = DataLoader(SpectrogramDataset(train_items, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=0, drop_last=False, pin_memory=PIN)
val_loader = DataLoader(SpectrogramDataset(val_items, eval_tf), batch_size=BATCH,
                        shuffle=False, num_workers=0, pin_memory=PIN)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), "labels:", tuple(yb.shape))   # (B, 1, 128, 128)

### 3. CNN from scratch


Architecture required by the course (slide 17): **4 blocks**, each = two 3×3 convolutions
(+ BatchNorm + ReLU) followed by a 2×2 max-pool, with channels doubling at each block:
**1 → 32 → 64 → 128 → 256** (the leading 1 = our single-channel image). Then a *Global
Average Pooling* summarises each map into one value, a *dropout* limits overfitting, and a
final linear layer votes for the emotion.

The repeated pattern is factored into a `ConvBlock` class, reused 4 times.

In [ ]:
class ConvBlock(nn.Module):
    '''Deux convolutions 3x3 -> BN + ReLU x2 -> MaxPool'''
    
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1) 
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.pool  = nn.MaxPool2d(2)                                      

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return self.pool(x)


class EmotionCNN(nn.Module):
    def __init__(self, n_classes, in_ch=1):
        super().__init__()
        self.block1 = ConvBlock(in_ch, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.block4 = ConvBlock(128, 256)
        self.gap     = nn.AdaptiveAvgPool2d(1)   
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(256, n_classes) 

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)            
        x = torch.flatten(x, 1)    
        x = self.dropout(x)
        return self.fc(x)          


model = EmotionCNN(n_classes=len(CLASSES), in_ch=1).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters : {total:,}")

# quick shape check
out = model(torch.randn(4, 1, IMG_SIZE, IMG_SIZE).to(device))
print("output :", tuple(out.shape), "(should be (4, %d))" % len(CLASSES))

In [ ]:
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parametre : {n_params:,}")

dummy = torch.randn(4, 1, IMG_SIZE, IMG_SIZE).to(device)  # 4 fausses images
out = model(dummy)
print("shape of the output :", out.shape)

### 4. Training loop
The dataset is imbalanced (more "anger" than "disgust"). The loss is weighted by the
inverse class frequency, otherwise the network mostly learns to predict the majority
class.

In [ ]:
counts = np.bincount([y for _, y, _ in train_items], minlength=len(CLASSES))
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * np.maximum(counts, 1)),
                             dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print({c: round(float(w), 2) for c, w in zip(CLASSES, class_weights)})


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    loss_sum, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return loss_sum / n, correct / n


def fit(model, optimizer, epochs, history, tag="scratch"):
    best_val, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        history.append({"phase": tag, "train_acc": tr_acc, "val_acc": va_acc,
                        "train_loss": tr_loss, "val_loss": va_loss})
        flag = ""
        if va_acc > best_val:
            best_val = va_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            flag = "  <-- best"
        print(f"[{tag}] epoch {epoch:2d}  train {tr_loss:.3f}/{tr_acc:.3f}   val {va_loss:.3f}/{va_acc:.3f}{flag}")
    if best_state is not None:
        model.load_state_dict(best_state)   # restore the best-on-validation weights
        print(f"restored best validation accuracy: {best_val:.3f}")
    return history

### Training

In [ ]:
history = []
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
fit(model, optimizer, EPOCHS, history)

### Saving the model for inference

We save :

- the class list in the right order
- the spectrogram parameters (`spec`)
- the input size and number of channels used by the transforms —

Everything goes into a single checkpoint dictionary, in the same format as
`transfer_learning.ipynb`, reloaded with `torch.load(MODEL_PATH)` then
`model.load_state_dict(ckpt["state_dict"])`.

In [ ]:
# same values as in prepare_*.ipynb (step 3: wav -> 3 s segments -> mel-spectrogram)
SR = 16000
SPEC = {
    "sr": SR,
    "duration": 3.0,             # length of one segment (s)
    "min_seg": 1.0,              # shorter trailing segments are dropped
    "n_fft": int(0.025 * SR),    # 25 ms window
    "hop": int(0.010 * SR),      # 10 ms hop
    "n_mels": 64,
}

checkpoint = {
    "state_dict": model.state_dict(),
    "classes": CLASSES,
    "spec": SPEC,
    "img_size": IMG_SIZE,
    "in_channels": 1,            # grayscale (the ResNet checkpoint has RGB / no this field)
    "norm": {"mean": [0.5], "std": [0.5]},
    "datasets": DATASETS,
}
torch.save(checkpoint, MODEL_PATH)
print("checkpoint saved to", MODEL_PATH)
print("classes:", CLASSES)

### 5. Learning curves

If the *train* curve keeps rising while validation stalls,it is overfitting

In [ ]:
import matplotlib.pyplot as plt

TRAIN_C, VAL_C = "#4571C4", "#D97757"
epochs = range(1, len(history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, key, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    ax.plot(epochs, [h[f"train_{key}"] for h in history], color=TRAIN_C, lw=2, label="train")
    ax.plot(epochs, [h[f"val_{key}"] for h in history], color=VAL_C, lw=2, label="validation")
    ax.set_title(title); ax.set_xlabel("epoch")
    ax.grid(alpha=0.25, lw=0.6); ax.spines[["top", "right"]].set_visible(False)
axes[1].set_ylim(0, 1)
axes[0].legend(frameon=False)
plt.show()

best = max(history, key=lambda h: h["val_acc"])
print(f"best validation accuracy : {best['val_acc']:.3f}  (chance = {1/len(CLASSES):.3f})")

### 6. Evaluation: per-class report and confusion matrix


In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, balanced_accuracy_score)

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_loader:
        y_pred += model(x.to(device)).argmax(1).cpu().tolist()
        y_true += y.tolist()

print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))
print(f"UAR (balanced accuracy) : {balanced_accuracy_score(y_true, y_pred):.3f}")

cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)), normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=45)
ax.set_title("Confusion matrix (validation, row-normalised)")
ax.set_xlabel("prediction"); ax.set_ylabel("ground truth"); ax.grid(False)
plt.show()